In [1]:
import os
import yaml
import copy
import time
import numpy as np
import pandas as pd
import xarray as xr

from xclim.indices import (
    standardized_precipitation_evapotranspiration_index,
    water_budget,
)

from xclim.indices.stats import (
    standardized_index_fit_params, 
    standardized_index
)

In [1]:
def process_vars(data):
    """
    Preprocess data variables for SPEI calculation.
    """
    
    # data["time"] = (pd.to_datetime(data["time"]).to_numpy().astype("datetime64[D]"))
    data["time"] = pd.to_datetime(data["time"]).to_numpy().astype("datetime64[ns]")
    first_date = data.time.min().dt.strftime("%Y-%m-%d").values.item()
    
    if not first_date.endswith("-01-01"):
        first_year = int(first_date[:4])
        data = data.sel(time=slice(str(first_year + 1), None))

    precip = data["precip"]
    tmin = data["tmin"]
    tmax = data["tmax"]
    
    return precip, tmin, tmax

# ============================================================== #
# Fairbanks
# ============================================================== #

stn = 'Fairbanks'
vars_keep = ["TREFHT", "TREFHTMX"]
base_dir = '/glade/campaign/ral/hap/ksha/EPRI_data/ERA5_daily/'

list_ds = []

for year in range(1958, 2026):

    # get data and variable
    fn_ERA5 = base_dir + f'{stn}_{year}.zarr'
    ds = xr.open_zarr(fn_ERA5)[vars_keep]
    
    # groupby objects (cheap)
    g_TREFHT = ds["TREFHT"].groupby("time.year")
    g_TREFHTMX = ds["TREFHTMX"].groupby("time.year")

    # annual maxima (vectorized)
    da_TREFHTMX_max = g_TREFHTMX.max("time", skipna=True).rename("T2_1h_max")
    da_TREFHT_max   = g_TREFHT.max("time", skipna=True).rename("T2_1d_max")

    # rolling maxima over each year (avoid groupby.map(lambda ...))
    da_TREFHT_7d_max = (
        ds["TREFHT"]
        .rolling(time=7, min_periods=7).mean()
        .groupby("time.year").max("time", skipna=True)
        .rename("T2_7d_max")
    )

    da_TREFHT_30d_max = (
        ds["TREFHT"]
        .rolling(time=30, min_periods=30).mean()
        .groupby("time.year").max("time", skipna=True)
        .rename("T2_30d_max")
    )
    
    # ============ #
    ds_merge = xr.merge([da_TREFHTMX_max, da_TREFHT_max, da_TREFHT_7d_max, da_TREFHT_30d_max])
    list_ds.append(ds_merge)
    
ds_all = xr.concat(list_ds, dim='year').rename({'year': 'valid_year'})
ds_all = ds_all.chunk({'valid_year': -1})

dir_stn = f'/glade/derecho/scratch/ksha/EPRI_data/METRICS/{stn}/'
save_name = dir_stn + "ERA5_STN.zarr"
ds_all.to_zarr(save_name, mode="w")
print(save_name)

# ============================================================== #
# Pituffik
# ============================================================== #

stn = 'Pituffik'
vars_keep = ["TREFHT", "TREFHTMX"]
base_dir = '/glade/campaign/ral/hap/ksha/EPRI_data/ERA5_daily/'

list_MDD = []
thres = 273.15
for year in range(1958, 2026):

    # get data and variable
    fn_ERA5 = base_dir + f'{stn}_{year}.zarr'
    ds = xr.open_zarr(fn_ERA5)[vars_keep]
    ds = ds.sel(time=slice(f"{year}-03-01", f"{year}-06-30"))
    mdd_mj = (ds["TREFHT"] - thres).clip(min=0).sum(dim="time")
    ds_MDD = mdd_mj.to_dataset(name="MDD").assign_coords(year=year).expand_dims("year")
    
    list_MDD.append(ds_MDD)

ds_MDD_all = xr.merge(list_MDD)

list_FT = []
Tf = 273.15  # K

vars_keep = ["TREFHT", "TREFHTMX", 'TREFHTMN']

for year in range(1958, 2026):

    fn_ERA5 = base_dir + f'{stn}_{year}.zarr'
    ds = xr.open_zarr(fn_ERA5)[vars_keep]

    # Restrict to the March–June season
    ds = ds.sel(time=slice(f"{year}-03-01", f"{year}-06-30"))

    # Freeze–Thaw day mask (1 if crossing, else 0)
    ft_day = ((ds["TREFHTMN"] < Tf) & (ds["TREFHTMX"] > Tf)).astype("int8")

    # Count days in the season
    ft_count = ft_day.sum(dim="time")

    ds_FT = ft_count.to_dataset(name="FT_days").assign_coords(year=year).expand_dims("year")
    list_FT.append(ds_FT)

ds_FT_all = xr.concat(list_FT, dim="year")

ds_all = xr.merge([ds_FT_all, ds_MDD_all])
ds_all = ds_all.rename({'year': 'valid_year'})
ds_all = ds_all.chunk({'valid_year': -1})

dir_stn = f'/glade/derecho/scratch/ksha/EPRI_data/METRICS/{stn}/'
save_name = dir_stn + 'ERA5_STN.zarr'
ds_all.to_zarr(save_name, mode='w')
print(save_name)

# ============================================================== #
# Guam
# ============================================================== #

stn = 'Guam'
base_dir = '/glade/campaign/ral/hap/ksha/EPRI_data/ERA5_daily/'

ds_collection = []

for year in range(1958, 2026):

    # get data and variable
    fn_ERA5 = base_dir + f'{stn}_{year}.zarr'
    ds_ERA5 = xr.open_zarr(fn_ERA5)[["PRECT", 'TREFHTMX', 'TREFHTMN']]
    
    # ----------------------------------------------- #
    ds_ERA5["PRECT"] = ds_ERA5["PRECT"] * 1000 /86400
    # ----------------------------------------------- #
    
    ds_ERA5["TREFHTMX"] -= 273.15
    ds_ERA5["TREFHTMN"] -= 273.15
    
    ds_collection.append(ds_ERA5)

ds_all = xr.concat(ds_collection, dim='time')

SPEI_03 = np.empty((12*(2025-1958+1)))
SPEI_03[...] = np.nan

SPEI_48 = np.empty((12*(2025-1958+1)))
SPEI_48[...] = np.nan

ds_subset = ds_all
lat_mid = lat_ref = ds_subset['latitude'].values
time_vals = ds_subset['time']

tmin = ds_subset['TREFHTMN'].values
tmax = ds_subset['TREFHTMX'].values
precip = ds_subset['PRECT'].values
precip[precip<0 ] = 0

ds = xr.Dataset(
    {
        "precip": (("time",), precip, {"units": "kg m-2 s-1"}),
        "tmin":   (("time",), tmin, {"units": "degC"}),
        "tmax":   (("time",), tmax, {"units": "degC"}),
    },
    coords={"time": time_vals, "lat": lat_mid}
)

for v in ("precip", "tmin", "tmax"):
    
    ds[v] = ds[v].assign_coords(lat=lat_mid)
    
    ds[v]["lat"].attrs = {
        "standard_name": "latitude",
        "units": "degrees_north", 
        "axis": "Y"
    }

precip, tmin, tmax = process_vars(ds)

# ------------------------------------------------------------ #
# Compute the daily water budget once for the full record
# ------------------------------------------------------------ #

wb = water_budget(
    pr=precip, tasmin=tmin, tasmax=tmax,
    method="HG85", lat=precip["lat"],
)
wb.attrs["units"] = "kg m-2 s-1"

# ------------------------------------------------------------ #
# Calibration period: 1968-01-01 to 1999-12-31
# Application period: 1958-01-01 to 2025-12-31 (full record)
# ------------------------------------------------------------ #
cal_start = "1968-01-01"
cal_end   = "1999-12-31"

dist   = "fisk"
method = "ML" # "APP", "PWM"

# ------------------------------------------------------------ #
# SPEI-24: fit params on calibration period, apply to full record
# ------------------------------------------------------------ #
params_03 = standardized_index_fit_params(
    wb.sel(time=slice(cal_start, cal_end)),
    freq="MS",
    window=3,
    dist=dist,
    method=method,
)

spei_03 = standardized_index(
    wb,
    freq="MS",
    window=3,
    dist=dist,
    method=method,
    params=params_03,
    zero_inflated=False,
    fitkwargs=None,
    cal_start=None,
    cal_end=None,
)

# ------------------------------------------------------------ #
# SPEI-48: fit params on calibration period, apply to full record
# ------------------------------------------------------------ #
params_48 = standardized_index_fit_params(
    wb.sel(time=slice(cal_start, cal_end)),
    freq="MS",
    window=48,
    dist=dist,
    method=method,
)

spei_48 = standardized_index(
    wb,
    freq="MS",
    window=48,
    dist=dist,
    method=method,
    params=params_48,
    zero_inflated=False,
    fitkwargs=None,
    cal_start=None,
    cal_end=None,
)

# ------------------------------------------------------------ #
# Store results
# ------------------------------------------------------------ #

SPEI_03[:] = spei_03.values
SPEI_48[:] = spei_48.values

time = pd.date_range("1958-01-01", "2025-12-01", freq="MS")

ds_SPEI = xr.Dataset(
    data_vars={
        "SPEI_03": (("time",), SPEI_03),
        "SPEI_48": (("time",), SPEI_48)
    },
    coords={
        "time": time,
    },
)

ds_SPEI_clean = ds_SPEI.where(ds_SPEI >= -4.5)

coarsened = ds_SPEI_clean.coarsen(time=12)  # 120 / 12 = 10 lead years

ds_SPEI_annual = xr.merge([
    coarsened.min(skipna=True).rename({v: f"{v}_min"  for v in ds_SPEI_clean.data_vars}),
    coarsened.mean(skipna=True).rename({v: f"{v}_mean" for v in ds_SPEI_clean.data_vars}),
]).rename({"time": "valid_year"}).assign_coords(valid_year=np.arange(1958, 2026))

ds_SPEI_annual = ds_SPEI_annual.interpolate_na(dim="valid_year", method="linear", max_gap=999)

ds_all = ds_SPEI_annual

dir_stn = f'/glade/derecho/scratch/ksha/EPRI_data/METRICS/{stn}/'
save_name = dir_stn + 'ERA5_STN.zarr'
ds_all.to_zarr(save_name, mode='w')
print(save_name)


# ============================================================== #
# Yuma PG
# ============================================================== #

stn = 'Yuma_PG'
vars_keep = ["PRECT",]
base_dir = '/glade/campaign/ral/hap/ksha/EPRI_data/ERA5_daily/'

ds_collection = []

for year in range(1958, 2026):

    # get data and variable
    fn_ERA5 = base_dir + f'{stn}_{year}.zarr'
    ds = xr.open_zarr(fn_ERA5)[vars_keep]

    # ---------------------------------- #
    ds["PRECT"] = ds["PRECT"] * 1000
    # ---------------------------------- #
    
    # ============ #
    # PRECT max and weekly max
    ds_TP_group = ds[["PRECT"]].groupby("time.year")
    ds_TP_max  = ds_TP_group.max(dim="time",  skipna=True)
    ds_TP_7d = ds_TP_group.map(lambda x: x.rolling(time=7, min_periods=7).mean().max(dim="time", skipna=True))
    
    # rename and merge ds_mean, ds_min, ds_max, ds_30d_max
    ds_TP_max = ds_TP_max.rename({v: f"{v}_1d_max" for v in ds_TP_max.data_vars})
    ds_TP_7d = ds_TP_7d.rename({v: f"{v}_7d_max"  for v in ds_TP_7d.data_vars})
    # ============ #
    
    ds_merge = xr.merge([ds_TP_max, ds_TP_7d])
    ds_collection.append(ds_merge)

ds_TP_max = xr.concat(ds_collection, dim='year').rename({'year': 'valid_year'})
ds_TP_max = ds_TP_max.chunk({'valid_year': -1})

ds_collection = []

for year in range(1958, 2026):

    # get data and variable
    fn_ERA5 = base_dir + f'{stn}_{year}.zarr'
    ds = xr.open_zarr(fn_ERA5)[vars_keep]

    # ---------------------------------- #
    ds["PRECT"] = ds["PRECT"] * 1000
    # ---------------------------------- #
    
    ds_collection.append(ds)

ds_all = xr.concat(ds_collection, dim='time')
da = ds_all["PRECT"].chunk({"time": -1})

clim_p999 = da.quantile(0.999, dim="time", skipna=True)
clim_p999 = clim_p999.load()

ds_event = ds_all > clim_p999
ds_event_group = ds_event.groupby("time.year")
ds_event_all = ds_event_group.sum(dim="time", skipna=True)
ds_event_all = ds_event_all.chunk({'year': -1})

ds_event_all = ds_event_all.rename({'PRECT': 'days_above_999p'})
ds_event_all = ds_event_all.rename({'year': 'valid_year'})

ds_collection = []

for year in range(1958, 2026):

    # get data and variable
    fn_ERA5 = base_dir + f'{stn}_{year}.zarr'
    ds_ERA5 = xr.open_zarr(fn_ERA5)[["PRECT", 'TREFHTMX', 'TREFHTMN']]
    
    # ----------------------------------------------- #
    ds_ERA5["PRECT"] = ds_ERA5["PRECT"] * 1000 /86400
    # ----------------------------------------------- #
    
    ds_ERA5["TREFHTMX"] -= 273.15
    ds_ERA5["TREFHTMN"] -= 273.15
    
    ds_collection.append(ds_ERA5)

ds_all = xr.concat(ds_collection, dim='time')

SPEI_24 = np.empty((12*(2025-1958+1)))
SPEI_24[...] = np.nan

SPEI_48 = np.empty((12*(2025-1958+1)))
SPEI_48[...] = np.nan

ds_subset = ds_all
lat_mid = lat_ref = ds_subset['latitude'].values
time_vals = ds_subset['time']

tmin = ds_subset['TREFHTMN'].values
tmax = ds_subset['TREFHTMX'].values
precip = ds_subset['PRECT'].values
precip[precip<0 ] = 0

ds = xr.Dataset(
    {
        "precip": (("time",), precip, {"units": "kg m-2 s-1"}),
        "tmin":   (("time",), tmin, {"units": "degC"}),
        "tmax":   (("time",), tmax, {"units": "degC"}),
    },
    coords={"time": time_vals, "lat": lat_mid}
)

for v in ("precip", "tmin", "tmax"):
    
    ds[v] = ds[v].assign_coords(lat=lat_mid)
    
    ds[v]["lat"].attrs = {
        "standard_name": "latitude",
        "units": "degrees_north", 
        "axis": "Y"
    }

precip, tmin, tmax = process_vars(ds)

# ------------------------------------------------------------ #
# Compute the daily water budget once for the full record
# ------------------------------------------------------------ #

wb = water_budget(
    pr=precip, tasmin=tmin, tasmax=tmax,
    method="HG85", lat=precip["lat"],
)
wb.attrs["units"] = "kg m-2 s-1"

# ------------------------------------------------------------ #
# Calibration period: 1968-01-01 to 1999-12-31
# Application period: 1958-01-01 to 2025-12-31 (full record)
# ------------------------------------------------------------ #
cal_start = "1968-01-01"
cal_end   = "1999-12-31"

dist   = "fisk"
method = "ML" # "APP", "PWM"

# ------------------------------------------------------------ #
# SPEI-24: fit params on calibration period, apply to full record
# ------------------------------------------------------------ #
params_24 = standardized_index_fit_params(
    wb.sel(time=slice(cal_start, cal_end)),
    freq="MS",
    window=24,
    dist=dist,
    method=method,
)

spei_24 = standardized_index(
    wb,
    freq="MS",
    window=24,
    dist=dist,
    method=method,
    params=params_24,
    zero_inflated=False,
    fitkwargs=None,
    cal_start=None,
    cal_end=None,
)

# ------------------------------------------------------------ #
# SPEI-48: fit params on calibration period, apply to full record
# ------------------------------------------------------------ #
params_48 = standardized_index_fit_params(
    wb.sel(time=slice(cal_start, cal_end)),
    freq="MS",
    window=48,
    dist=dist,
    method=method,
)

spei_48 = standardized_index(
    wb,
    freq="MS",
    window=48,
    dist=dist,
    method=method,
    params=params_48,
    zero_inflated=False,
    fitkwargs=None,
    cal_start=None,
    cal_end=None,
)

# ------------------------------------------------------------ #
# Store results
# ------------------------------------------------------------ #

SPEI_24[:] = spei_24.values
SPEI_48[:] = spei_48.values

time = pd.date_range("1958-01-01", "2025-12-01", freq="MS")

ds_SPEI = xr.Dataset(
    data_vars={
        "SPEI_24": (("time",), SPEI_24),
        "SPEI_48": (("time",), SPEI_48)
    },
    coords={
        "time": time,
    },
)

ds_SPEI_clean = ds_SPEI.where(ds_SPEI >= -4.5)

coarsened = ds_SPEI_clean.coarsen(time=12)  # 120 / 12 = 10 lead years

ds_SPEI_annual = xr.merge([
    coarsened.min(skipna=True).rename({v: f"{v}_min"  for v in ds_SPEI_clean.data_vars}),
    coarsened.mean(skipna=True).rename({v: f"{v}_mean" for v in ds_SPEI_clean.data_vars}),
]).rename({"time": "valid_year"}).assign_coords(valid_year=np.arange(1958, 2026))

ds_SPEI_annual = ds_SPEI_annual.interpolate_na(dim="valid_year", method="linear", max_gap=999)

ds_all = xr.merge([ds_TP_max, ds_event_all, ds_SPEI_annual])

dir_stn = f'/glade/derecho/scratch/ksha/EPRI_data/METRICS/{stn}/'
save_name = dir_stn + 'ERA5_STN.zarr'
ds_all.to_zarr(save_name, mode='w')
print(save_name)

# ============================================================== #
# Fort Bragg
# ============================================================== #

stn = 'Fort_Bragg'
vars_keep = ["PRECT",]
base_dir = '/glade/campaign/ral/hap/ksha/EPRI_data/ERA5_daily/'

ds_collection = []

for year in range(1958, 2026):

    # get data and variable
    fn_ERA5 = base_dir + f'{stn}_{year}.zarr'
    ds = xr.open_zarr(fn_ERA5)[vars_keep]

    # ---------------------------------- #
    ds["PRECT"] = ds["PRECT"] * 1000
    # ---------------------------------- #
    
    # ============ #
    # PRECT max and weekly max
    ds_TP_group = ds[["PRECT"]].groupby("time.year")
    ds_TP_max  = ds_TP_group.max(dim="time",  skipna=True)
    ds_TP_3d = ds_TP_group.map(lambda x: x.rolling(time=3, min_periods=3).mean().max(dim="time", skipna=True))
    ds_TP_5d = ds_TP_group.map(lambda x: x.rolling(time=5, min_periods=5).mean().max(dim="time", skipna=True))
    
    # rename and merge ds_mean, ds_min, ds_max, ds_30d_max
    ds_TP_max = ds_TP_max.rename({v: f"{v}_1d_max" for v in ds_TP_max.data_vars})
    ds_TP_3d = ds_TP_3d.rename({v: f"{v}_3d_max"  for v in ds_TP_3d.data_vars})
    ds_TP_5d = ds_TP_5d.rename({v: f"{v}_5d_max"  for v in ds_TP_5d.data_vars})
    # ============ #
    
    ds_merge = xr.merge([ds_TP_max, ds_TP_3d, ds_TP_5d])
    ds_collection.append(ds_merge)

ds_TP_max = xr.concat(ds_collection, dim='year').rename({'year': 'valid_year'})
ds_TP_max = ds_TP_max.chunk({'valid_year': -1})

ds_collection = []

for year in range(1958, 2026):

    # get data and variable
    fn_ERA5 = base_dir + f'{stn}_{year}.zarr'
    ds_ERA5 = xr.open_zarr(fn_ERA5)[["PRECT", 'TREFHTMX', 'TREFHTMN']]
    
    # ----------------------------------------------- #
    ds_ERA5["PRECT"] = ds_ERA5["PRECT"] * 1000 /86400
    # ----------------------------------------------- #
    
    ds_ERA5["TREFHTMX"] -= 273.15
    ds_ERA5["TREFHTMN"] -= 273.15
    
    ds_collection.append(ds_ERA5)

ds_all = xr.concat(ds_collection, dim='time')

SPEI_09 = np.empty((12*(2025-1958+1)))
SPEI_09[...] = np.nan

SPEI_48 = np.empty((12*(2025-1958+1)))
SPEI_48[...] = np.nan

ds_subset = ds_all
lat_mid = lat_ref = ds_subset['latitude'].values
time_vals = ds_subset['time']

tmin = ds_subset['TREFHTMN'].values
tmax = ds_subset['TREFHTMX'].values
precip = ds_subset['PRECT'].values
precip[precip<0 ] = 0

ds = xr.Dataset(
    {
        "precip": (("time",), precip, {"units": "kg m-2 s-1"}),
        "tmin":   (("time",), tmin, {"units": "degC"}),
        "tmax":   (("time",), tmax, {"units": "degC"}),
    },
    coords={"time": time_vals, "lat": lat_mid}
)

for v in ("precip", "tmin", "tmax"):
    
    ds[v] = ds[v].assign_coords(lat=lat_mid)
    
    ds[v]["lat"].attrs = {
        "standard_name": "latitude",
        "units": "degrees_north", 
        "axis": "Y"
    }

precip, tmin, tmax = process_vars(ds)

# ------------------------------------------------------------ #
# Compute the daily water budget once for the full record
# ------------------------------------------------------------ #

wb = water_budget(
    pr=precip, tasmin=tmin, tasmax=tmax,
    method="HG85", lat=precip["lat"],
)
wb.attrs["units"] = "kg m-2 s-1"

# ------------------------------------------------------------ #
# Calibration period: 1968-01-01 to 1999-12-31
# Application period: 1958-01-01 to 2025-12-31 (full record)
# ------------------------------------------------------------ #
cal_start = "1968-01-01"
cal_end   = "1999-12-31"

dist   = "fisk"
method = "ML" # "APP", "PWM"

# ------------------------------------------------------------ #
# SPEI-24: fit params on calibration period, apply to full record
# ------------------------------------------------------------ #
params_09 = standardized_index_fit_params(
    wb.sel(time=slice(cal_start, cal_end)),
    freq="MS",
    window=9,
    dist=dist,
    method=method,
)

spei_09 = standardized_index(
    wb,
    freq="MS",
    window=9,
    dist=dist,
    method=method,
    params=params_09,
    zero_inflated=False,
    fitkwargs=None,
    cal_start=None,
    cal_end=None,
)

# ------------------------------------------------------------ #
# SPEI-48: fit params on calibration period, apply to full record
# ------------------------------------------------------------ #
params_48 = standardized_index_fit_params(
    wb.sel(time=slice(cal_start, cal_end)),
    freq="MS",
    window=48,
    dist=dist,
    method=method,
)

spei_48 = standardized_index(
    wb,
    freq="MS",
    window=48,
    dist=dist,
    method=method,
    params=params_48,
    zero_inflated=False,
    fitkwargs=None,
    cal_start=None,
    cal_end=None,
)

# ------------------------------------------------------------ #
# Store results
# ------------------------------------------------------------ #

SPEI_09[:] = spei_09.values
SPEI_48[:] = spei_48.values

time = pd.date_range("1958-01-01", "2025-12-01", freq="MS")

ds_SPEI = xr.Dataset(
    data_vars={
        "SPEI_09": (("time",), SPEI_09),
        "SPEI_48": (("time",), SPEI_48)
    },
    coords={
        "time": time,
    },
)

ds_SPEI_clean = ds_SPEI.where(ds_SPEI >= -4.5)

coarsened = ds_SPEI_clean.coarsen(time=12)  # 120 / 12 = 10 lead years

ds_SPEI_annual = xr.merge([
    coarsened.min(skipna=True).rename({v: f"{v}_min"  for v in ds_SPEI_clean.data_vars}),
    coarsened.mean(skipna=True).rename({v: f"{v}_mean" for v in ds_SPEI_clean.data_vars}),
]).rename({"time": "valid_year"}).assign_coords(valid_year=np.arange(1958, 2026))

ds_SPEI_annual = ds_SPEI_annual.interpolate_na(dim="valid_year", method="linear", max_gap=999)

ds_all = xr.merge([ds_TP_max, ds_SPEI_annual])

dir_stn = f'/glade/derecho/scratch/ksha/EPRI_data/METRICS/{stn}/'
save_name = dir_stn + 'ERA5_STN.zarr'
ds_all.to_zarr(save_name, mode='w')
print(save_name)


/glade/derecho/scratch/ksha/EPRI_data/METRICS/Fairbanks/ERA5_STN.zarr
/glade/derecho/scratch/ksha/EPRI_data/METRICS/Pituffik/ERA5_STN.zarr
/glade/derecho/scratch/ksha/EPRI_data/METRICS/Guam/ERA5_STN.zarr
/glade/derecho/scratch/ksha/EPRI_data/METRICS/Yuma_PG/ERA5_STN.zarr
/glade/derecho/scratch/ksha/EPRI_data/METRICS/Fort_Bragg/ERA5_STN.zarr


In [5]:
xr.open_zarr('/glade/derecho/scratch/ksha/EPRI_data/METRICS/Guam/ERA5_STN.zarr')

<xarray.Dataset> Size: 3kB
Dimensions:       (valid_year: 68)
Coordinates:
  * valid_year    (valid_year) int64 544B 1958 1959 1960 1961 ... 2023 2024 2025
Data variables:
    SPEI_03_mean  (valid_year) float64 544B dask.array<chunksize=(68,), meta=np.ndarray>
    SPEI_03_min   (valid_year) float64 544B dask.array<chunksize=(68,), meta=np.ndarray>
    SPEI_48_mean  (valid_year) float64 544B dask.array<chunksize=(68,), meta=np.ndarray>
    SPEI_48_min   (valid_year) float64 544B dask.array<chunksize=(68,), meta=np.ndarray>

In [5]:
# ============================================================== #
# Save all
# ============================================================== #
stn_names = ['Fairbanks', 'Pituffik', 'Guam', 'Yuma_PG', 'Fort_Bragg']

list_ds = []
for stn in stn_names:
    fn = f'/glade/derecho/scratch/ksha/EPRI_data/METRICS/{stn}/' + "ERA5_STN.zarr"
    ds = xr.open_zarr(fn)
    try:
        ds = ds.drop_vars(['lat', 'lon'], errors='ignore')
        ds = ds.drop_vars(['latitude', 'longitude'], errors='ignore')
    except:
        pass
        
    # keep valid_year shared across stations (don't prefix it)
    if "valid_year" in ds.data_vars:
        ds = ds.set_coords("valid_year")

    # prefix every remaining data variable with the station name
    ds = ds.rename({v: f'{stn}_{v}' for v in ds.data_vars})

    list_ds.append(ds)

ds_all = xr.merge(list_ds)

ds_all = ds_all.drop_vars(['quantile',])

save_name = '/glade/derecho/scratch/ksha/EPRI_data/METRICS/STN_ERA5_ALL.zarr'
ds_all.to_zarr(save_name, mode='w')
print(save_name)

/glade/derecho/scratch/ksha/EPRI_data/METRICS/STN_ERA5_ALL.zarr


In [6]:
ds_all

<xarray.Dataset> Size: 11kB
Dimensions:                  (valid_year: 68)
Coordinates:
  * valid_year               (valid_year) int64 544B 1958 1959 ... 2024 2025
Data variables: (12/24)
    Fairbanks_T2_1d_max      (valid_year) float32 272B dask.array<chunksize=(68,), meta=np.ndarray>
    Fairbanks_T2_1h_max      (valid_year) float32 272B dask.array<chunksize=(68,), meta=np.ndarray>
    Fairbanks_T2_30d_max     (valid_year) float32 272B dask.array<chunksize=(68,), meta=np.ndarray>
    Fairbanks_T2_7d_max      (valid_year) float32 272B dask.array<chunksize=(68,), meta=np.ndarray>
    Pituffik_FT_days         (valid_year) int64 544B dask.array<chunksize=(68,), meta=np.ndarray>
    Pituffik_MDD             (valid_year) float32 272B dask.array<chunksize=(68,), meta=np.ndarray>
    ...                       ...
    Fort_Bragg_PRECT_3d_max  (valid_year) float32 272B dask.array<chunksize=(68,), meta=np.ndarray>
    Fort_Bragg_PRECT_5d_max  (valid_year) float32 272B dask.array<chunksize=(68,), meta=np.ndarray>
    Fort_Bragg_SPEI_09_mean  (valid_year) float64 544B dask.array<chunksize=(68,), meta=np.ndarray>
    Fort_Bragg_SPEI_09_min   (valid_year) float64 544B dask.array<chunksize=(68,), meta=np.ndarray>
    Fort_Bragg_SPEI_48_mean  (valid_year) float64 544B dask.array<chunksize=(68,), meta=np.ndarray>
    Fort_Bragg_SPEI_48_min   (valid_year) float64 544B dask.array<chunksize=(68,), meta=np.ndarray>
Attributes:
    long_name:   Maximum temperature at 2 metres since previous post-processing
    short_name:  mx2t
    units:       K